# Week 4 Workshop: Data Filtering & Selection
## Traffic Accident Vehicles in Colombia

Practice filtering techniques to answer analytical questions about vehicles involved in traffic accidents.

**Duration:** ~2 hours

**Objectives:**
- Filter rows using comparison operators
- Combine conditions with & (AND), | (OR), ~ (NOT)
- Use .isin(), .between(), .str.contains()
- Use .query() for clean syntax
- Translate analytical questions into filter code

---

## Setup
Run this cell to load the dataset and see the overview.

In [ ]:
import pandas as pd
import numpy as np

# Load the Traffic Accident Vehicles dataset
df = pd.read_csv('../data/vehiculos_accidentes (1).csv')

# Quick overview
print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nVehicle types: {df['tipo_vehiculo'].nunique()} unique")
print(f"Departments: {df['departamento_accidente'].nunique()} unique")
print(f"Severity levels: {df['gravedad_accidente'].unique().tolist()}")
print(f"Brands: {df['marca_vehiculo'].nunique()} unique")
print(f"\nTop 5 vehicle types:")
print(df['tipo_vehiculo'].value_counts().head())
df.head(3)

Dataset: 20000 rows x 9 columns

Columns: ['marca_vehiculo', 'modelo_vehiculo', 'tipo_vehiculo', 'edad_vehiculo', 'fecha_accidente', 'gravedad_accidente', 'departamento_accidente', 'municipio_accidente', 'autoridad_de_transito']

Vehicle types: 16 unique
Departments: 29 unique
Severity levels: ['CON HERIDOS', 'CON MUERTOS']
Brands: 159 unique

Top 5 vehicle types:
tipo_vehiculo
MOTOCICLETA    11301
AUTOMOVIL       4185
CAMIONETA       1777
BUS              840
CAMION           618
Name: count, dtype: int64


,marca_vehiculo,modelo_vehiculo,tipo_vehiculo,edad_vehiculo,fecha_accidente,gravedad_accidente,departamento_accidente,municipio_accidente,autoridad_de_transito
0,AKT,2026,MOTOCICLETA,1.0,12/2025,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
1,BAJAJ,2022,MOTOCICLETA,4.0,12/2025,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
2,RENAULT,2021,AUTOMOVIL,5.0,12/2025,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA


---

## Part 1: Single Condition Filters (20 minutes)

Practice the basic filtering pattern: `df[df['column'] operator value]`

### Task 1.1: Filter by vehicle type
Get all records where the vehicle is a **MOTOCICLETA** (motorcycle).

Print the number of rows and display the first 5.

In [ ]:
# Filter motorcycles
motos = df[df['tipo_vehiculo'] == 'MOTOCICLETA']

print("Total motocicletas:", len(motos))
motos.head(5)


### Task 1.2: Filter by comparison
Find vehicles with age greater than 15 years (`edad_vehiculo > 15`).

**Question:** How many old vehicles were involved in accidents? What are the most common brands among them?

In [2]:
# Vehicles older than 15 years
old_vehicles = df[df['edad_vehiculo'] > 15]

print("Vehículos con más de 15 años:", len(old_vehicles))

# Most common brands
old_vehicles['marca_vehiculo'].value_counts().head()

Vehículos con más de 15 años: 3369


marca_vehiculo
CHEVROLET    864
HYUNDAI      336
RENAULT      243
YAMAHA       195
SUZUKI       190
Name: count, dtype: int64

### Task 1.3: Filter by exact text
Get all accidents that occurred in the department **ATLANTICO**.

Sort the results by `fecha_accidente` and display the first 10 rows.

In [3]:
# Accidents in Atlantico
atlantico = df[df['departamento_accidente'] == 'ATLANTICO']

# Sort by date
atlantico = atlantico.sort_values('fecha_accidente')

atlantico.head(10)

,marca_vehiculo,modelo_vehiculo,tipo_vehiculo,edad_vehiculo,fecha_accidente,gravedad_accidente,departamento_accidente,municipio_accidente,autoridad_de_transito
19948,AKT,2016,MOTOCICLETA,10.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19939,AKT,2019,MOTOCICLETA,7.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19940,BAJAJ,2010,MOTOCICLETA,16.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19941,BAJAJ,2022,MOTOCICLETA,4.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19942,BAJAJ,2023,MOTOCICLETA,4.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19947,AKT,2023,MOTOCICLETA,3.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19944,TOYOTA,2020,CAMIONETA,6.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19945,RENAULT,2007,AUTOMOVIL,19.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19946,AKT,2023,MOTOCICLETA,4.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA
19943,RENAULT,2022,AUTOMOVIL,4.0,12/2022,CON HERIDOS,ATLANTICO,BARRANQUILLA,STRIA DTAL TTO BARRANQUILLA


### Task 1.4: Filter with not equal
Get all records where the severity is NOT "CON HERIDOS" (i.e., get only the fatal accidents).

Print the count and compare it to the total dataset size.

In [4]:
print(f"Total records: {len(df)}")

fatal = df[df['gravedad_accidente'] != 'CON HERIDOS']

print(f"Fatal accidents: {len(fatal)}")
print(f"Percentage: {len(fatal) / len(df) * 100:.1f}%")

Total records: 20000
Fatal accidents: 1352
Percentage: 6.8%


---

## Part 2: Combining Conditions (30 minutes)

Combine multiple conditions to answer more specific questions.

**Rules:**
- Use `&` (AND), `|` (OR), `~` (NOT) - NOT `and`, `or`, `not`
- Always wrap each condition in parentheses

### Task 2.1: Two AND conditions
Find motorcycles (`MOTOCICLETA`) involved in fatal accidents (`CON MUERTOS`).

**Question:** How many fatal motorcycle accidents are in the dataset?

In [6]:
fatal_moto = df[
    (df['tipo_vehiculo'] == 'MOTOCICLETA') &
    (df['gravedad_accidente'] == 'CON MUERTOS')
]

print("Accidentes fatales con motocicletas:", len(fatal_moto))

Accidentes fatales con motocicletas: 758


### Task 2.2: Three AND conditions
Find **AUTOMOVIL** vehicles, in **BOGOTA D.C.**, with severity **CON HERIDOS**.

Show the brand (`marca_vehiculo`) and model year (`modelo_vehiculo`) columns of the results.

In [7]:
result = df[
    (df['tipo_vehiculo'] == 'AUTOMOVIL') &
    (df['departamento_accidente'] == 'BOGOTA D.C.') &
    (df['gravedad_accidente'] == 'CON HERIDOS')
]

result[['marca_vehiculo', 'modelo_vehiculo']]

,marca_vehiculo,modelo_vehiculo
7357,RENAULT,2007
7375,RENAULT,2023
7383,CITROEN,1994
7406,RENAULT,2007
7439,DAEWOO,2000
...,...,...
13457,RENAULT,2018
13468,RENAULT,2012
13628,RENAULT,2008
13630,RENAULT,2021


### Task 2.3: OR and .isin()
Find all accidents involving either **CHEVROLET**, **TOYOTA**, or **MAZDA** vehicles.

Try both approaches:
1. Using `|` (OR) operators
2. Using `.isin()`

Verify both give the same number of rows.

In [8]:
# YOUR CODE HERE

# Approach 1: Using | (OR)
brand_filter1 = df[
    (df['marca_vehiculo'] == 'CHEVROLET') |
    (df['marca_vehiculo'] == 'TOYOTA') |
    (df['marca_vehiculo'] == 'MAZDA')
]

# Approach 2: Using .isin()
brand_filter2 = df[
    df['marca_vehiculo'].isin(['CHEVROLET', 'TOYOTA', 'MAZDA'])
]

# Verify both have the same number of rows
print(len(brand_filter1))
print(len(brand_filter2))

3104
3104


### Task 2.4: Mixed AND + OR
Find accidents in **ANTIOQUIA** or **VALLE DEL CAUCA** where the vehicle is older than 10 years.

**Hint:** Combine `.isin()` with another condition:
```python
df[(df['col'].isin([...]) & (df['other_col'] > value)]
```

In [9]:
result = df[
    (df['departamento_accidente'].isin(['ANTIOQUIA','VALLE DEL CAUCA'])) &
    (df['edad_vehiculo'] > 10)
]

result.head()

,marca_vehiculo,modelo_vehiculo,tipo_vehiculo,edad_vehiculo,fecha_accidente,gravedad_accidente,departamento_accidente,municipio_accidente,autoridad_de_transito
68,BAJAJ,2010,MOTOCICLETA,17.0,12/2025,CON HERIDOS,ANTIOQUIA,APARTADO,STRIA MCPAL TTEyTTO APARTADO
69,YAMAHA,2016,MOTOCICLETA,11.0,12/2025,CON HERIDOS,ANTIOQUIA,APARTADO,STRIA MCPAL TTEyTTO APARTADO
71,YAMAHA,2014,MOTOCICLETA,12.0,12/2025,CON HERIDOS,ANTIOQUIA,APARTADO,STRIA MCPAL TTEyTTO APARTADO
106,FREIGHTLINER,2012,TRACTOCAMION,13.0,12/2025,CON MUERTOS,VALLE DEL CAUCA,BUENAVENTURA,DIRECCION DE TTOYTTE POLICIA NACIONAL - DITRA
107,FREIGHTLINER,2012,TRACTOCAMION,14.0,12/2025,CON MUERTOS,VALLE DEL CAUCA,BUENAVENTURA,DIRECCION DE TTOYTTE POLICIA NACIONAL - DITRA


### Task 2.5: NOT with .isin()
Get all accident records EXCLUDING motorcycles (`MOTOCICLETA`) and buses (`BUS`).

**Pattern:** `df[~df['col'].isin([list])]`

Print the count and the remaining vehicle types.

In [10]:
filtered = df[
    ~df['tipo_vehiculo'].isin(['MOTOCICLETA', 'BUS'])
]

print("Total registros:", len(filtered))

filtered['tipo_vehiculo'].value_counts()

Total registros: 7859


tipo_vehiculo
AUTOMOVIL        4185
CAMIONETA        1777
CAMION            618
CAMPERO           496
TRACTOCAMION      292
MICROBUS          208
BUSETA            135
VOLQUETA          102
MOTOCARRO          38
MONTACARGAS         2
CICLOMOTOR          2
TRACTOR             2
CUATRIMOTO          1
MAQ. AGRICOLA       1
Name: count, dtype: int64

---

## Part 3: Convenience Methods (20 minutes)

Practice .between(), .str.contains(), .str.startswith(), and .query() for common filtering patterns.

### Task 3.1: .between() for model year
Find vehicles with model year between **2015 and 2020** (inclusive).

**Pattern:** `df[df['col'].between(low, high)]`

Print how many records match and show the distribution of model years in the result.

In [11]:
model_range = df[df['modelo_vehiculo'].between(2015, 2020)]

print("Total registros:", len(model_range))

model_range['modelo_vehiculo'].value_counts().sort_index()

Total registros: 5999


modelo_vehiculo
2015    1026
2016    1003
2017     923
2018     863
2019    1022
2020    1162
Name: count, dtype: int64

### Task 3.2: .between() for age range
Get vehicles aged **0 to 5 years** (relatively new vehicles).

Print the count and the percentage of the total dataset.

In [12]:
new_vehicles = df[df['edad_vehiculo'].between(0,5)]

print("Cantidad:", len(new_vehicles))
print("Porcentaje:", len(new_vehicles)/len(df)*100)

Cantidad: 7123
Porcentaje: 35.615


### Task 3.3: .str.contains() for text search
Find transit authorities (`autoridad_de_transito`) whose name contains **"BOGOTA"**.

Print the unique authority names that match.

**Remember:** Always add `na=False` to handle potential NaN values.

In [13]:
bogota_authorities = df[
    df['autoridad_de_transito'].str.contains('BOGOTA', na=False)
]

bogota_authorities['autoridad_de_transito'].unique()

<StringArray>
['SECRETARIA DISTRITAL DE MOVILIDAD DE BOGOTA']
Length: 1, dtype: str

### Task 3.4: .str.startswith()
Find vehicle brands that start with **"CH"** (Chevrolet, Changan, Chery, etc.).

Print the unique brand names that match and the total number of records.

**Pattern:** `df[df['col'].str.startswith('text', na=False)]`

In [14]:
brands_ch = df[
    df['marca_vehiculo'].str.startswith('CH', na=False)
]

print("Total registros:", len(brands_ch))
print("Marcas encontradas:")
print(brands_ch['marca_vehiculo'].unique())

Total registros: 2348
Marcas encontradas:
<StringArray>
['CHANGAN', 'CHEVROLET', 'CHERY', 'CHANA', 'CHANGHE']
Length: 5, dtype: str


### Task 3.5: .query() method
Rewrite this boolean indexing filter using `.query()`:

```python
df[(df['edad_vehiculo'] > 10) & (df['departamento_accidente'] == 'ANTIOQUIA') & (df['tipo_vehiculo'] == 'MOTOCICLETA')]
```

Verify both approaches give the same number of rows.

In [15]:
# Boolean indexing version (given):
result_bool = df[
    (df['edad_vehiculo'] > 10) &
    (df['departamento_accidente'] == 'ANTIOQUIA') &
    (df['tipo_vehiculo'] == 'MOTOCICLETA')
]

result_query = df.query(
    "edad_vehiculo > 10 and departamento_accidente == 'ANTIOQUIA' and tipo_vehiculo == 'MOTOCICLETA'"
)

print(f"Boolean indexing: {len(result_bool)} rows")
print(f".query(): {len(result_query)} rows")



Boolean indexing: 999 rows
.query(): 999 rows


---

## Part 4: Analytical Questions (30 minutes)

Now use your filtering skills to answer real questions about traffic safety in Colombia. For each question:
1. Write the filter
2. Display relevant columns or value counts
3. Write a 1-2 sentence interpretation of the result

### Question 4.1: Fatal accidents by vehicle type
Which vehicle type is most involved in **fatal accidents** (`CON MUERTOS`)?

Filter for fatal accidents and count by `tipo_vehiculo`. Sort from highest to lowest.

In [16]:
fatal = df[df['gravedad_accidente'] == 'CON MUERTOS']

fatal['tipo_vehiculo'].value_counts()

tipo_vehiculo
MOTOCICLETA      758
AUTOMOVIL        154
CAMIONETA        118
TRACTOCAMION     108
CAMION           103
BUS               54
CAMPERO           20
VOLQUETA          14
MICROBUS          13
BUSETA             9
MAQ. AGRICOLA      1
Name: count, dtype: int64

**Your interpretation:** *(Write 1-2 sentences about what you observe)*


### Question 4.2: Motorcycle accidents by department
Which departments have the most **motorcycle** accidents?

Filter for `MOTOCICLETA` and count by `departamento_accidente`. Show the top 10.

In [17]:
motos = df[df['tipo_vehiculo'] == 'MOTOCICLETA']

motos['departamento_accidente'].value_counts().head(10)

departamento_accidente
ANTIOQUIA          3992
BOGOTA D.C.        1599
VALLE DEL CAUCA    1568
SANTANDER           562
ATLANTICO           387
TOLIMA              329
CUNDINAMARCA        320
CALDAS              319
RISARALDA           319
META                311
Name: count, dtype: int64

**Your interpretation:** *(Write 1-2 sentences)*


### Question 4.3: Vehicle age and severity
Are older vehicles more likely to be in **severe** (fatal) accidents?

Calculate the average `edad_vehiculo` for each `gravedad_accidente` level.

**Hint:** Filter for each severity level separately and compute `.mean()`, or use `df.groupby('gravedad_accidente')['edad_vehiculo'].mean()`.

In [18]:
df.groupby('gravedad_accidente')['edad_vehiculo'].mean()

gravedad_accidente
CON HERIDOS     9.642999
CON MUERTOS    10.815089
Name: edad_vehiculo, dtype: float64

**Your interpretation:** *(Write 1-2 sentences)*


### Question 4.4: Your own question
Design and answer your own analytical question about the traffic accident data.

Ideas:
- Which brands are most involved in fatal accidents in a specific department?
- How do accident patterns differ between Medellin and Bogota?
- What is the age profile of motorcycles vs. cars in accidents?
- Which municipalities have the most accidents with heavy vehicles (CAMION, TRACTOCAMION)?

**Your question:** *(Write it here)*


In [19]:
fatal = df[df['gravedad_accidente'] == 'CON MUERTOS']

fatal['marca_vehiculo'].value_counts().head(10)

marca_vehiculo
BAJAJ            195
YAMAHA           146
CHEVROLET        129
AKT              105
HONDA             98
SUZUKI            90
KENWORTH          55
RENAULT           47
INTERNATIONAL     40
TVS               37
Name: count, dtype: int64

**Your interpretation:** *(Write 1-2 sentences)*


---

## Part 5: Reflection

Answer these questions about your experience with data filtering.

### Reflection 1
Which filtering method did you find most useful: boolean indexing, .isin(), .between(), .str.contains(), or .query()? Why?

**El método que encontré más útil fue .isin().
Es más fácil de usar cuando se quiere buscar varios valores al mismo tiempo y hace que el código sea más corto y claro.**


### Reflection 2
What questions do you want to answer about **YOUR** project dataset using filtering? List at least 3 questions.

**Preguntas que me gustaría responder en mi proyecto:

1. ¿Qué tipo de vehículo está más involucrado en accidentes graves?

2. ¿Qué departamentos tienen mayor número de accidentes?

3. ¿Los vehículos más antiguos tienen más probabilidades de estar en accidentes graves?**


### Reflection 3
Why is it important to clean the data (Week 3) BEFORE filtering it (Week 4)? What problems could arise if you filter dirty data?

**Es importante limpiar los datos antes de filtrarlos porque si los datos tienen errores, valores vacíos o nombres escritos de forma diferente, los filtros pueden dar resultados incorrectos.

Por ejemplo, si "BOGOTA" está escrito de varias formas diferentes, el filtro podría ignorar algunos registros y el análisis sería incorrecto. **


---

## Summary

In this workshop you practiced:

| Skill | Methods |
|-------|--------|
| Single conditions | `==`, `!=`, `>`, `<`, `>=`, `<=` |
| Combining conditions | `&` (AND), `|` (OR), `~` (NOT) |
| Match a list | `.isin([list])` |
| Numeric range | `.between(low, high)` |
| Text pattern | `.str.contains()`, `.str.startswith()` |
| Clean syntax | `.query('expression')` |

These skills are essential for Milestone 1: you need to filter your project dataset to focus on specific subsets of data for analysis.

**Next week:** Exploratory Data Analysis (EDA) - using GroupBy and statistics to discover patterns.

---

*Week 4 - Data Analytics Course - Universidad Cooperativa de Colombia*